In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [ ]:
df = pd.read_excel("/content/ENB2012_data.xlsx")

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
X = df.drop(['Y1','Y2'],axis=1)
y = df['Y1']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape

In [ ]:
y_test.shape

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Converting into Tensors

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)
X_test_tensor = torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

# TensorDataset & DataLoader

In [ ]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader (test_dataset,batch_size=32)

Define ANN model

In [ ]:
class ANN(nn.Module):

  def __init__(self):

    super(ANN,self).__init__()

    self.model = nn.Sequential(
        nn.Linear(X_train.shape[1],8),
        nn.ReLU(),


        nn.Linear(8,8),
        nn.ReLU(),

        # output
        nn.Linear(8,1)
    )

  def forward(self,x):
    return self.model(x)

In [ ]:
model = ANN()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
epochs = 2000
train_losses = []
val_losses = []

for epoch in range(epochs):
  model.train()

  running_loss = 0.0
  for batch_features,batch_labels in train_loader:

    outputs = model(batch_features)

    optimizer.zero_grad()
    loss = criterion(outputs,batch_labels)
    loss.backward()

    optimizer.step()

    running_loss = running_loss + loss.item()

  total_loss = running_loss / len(train_loader)
  train_losses.append(total_loss)


  # val loss
  model.eval()

  running_val_loss = 0.0

  with torch.no_grad():

    for batch_features,batch_labels in test_loader:
      # Corrected typo: 'outputs' instead of 'ouputs'
      val_outputs = model(batch_features)
      # Used val_outputs for criterion and running_val_loss for accumulation
      loss = criterion(val_outputs,batch_labels)
      running_val_loss = running_val_loss + loss.item()

    # Used running_val_loss for total_val_loss calculation
    total_val_loss = running_val_loss / len(test_loader)
    val_losses.append(total_val_loss)

    print(f"Epcoh : {epoch + 1}/{epochs} , Training Loss : {total_loss}, Validation Loss : {total_val_loss}")



Evaluate the model

In [ ]:
model.eval()

with torch.no_grad():
  train_preds = model(X_train_tensor)
  test_preds = model(X_test_tensor)

  train_mse = criterion(train_preds,y_train_tensor)
  test_mse = criterion(test_preds,y_test_tensor)

  print(f"Training MSE Score : {train_mse}")
  print(f"Test MSE Score : {test_mse}")

In [ ]:
print(f"R2 Score : {r2_score(y_test,test_preds)}")